# ROME Submission Notebook

This notebook follows the provided ROME demo structure, runs the mux hierarchy, and extends it with a ripple-carry adder hierarchy.
Keep this notebook inside `HW_ROME/` so generated files stay in the submission folder.


In [1]:
#@title Setting up the notebook

### Installing dependencies
!pip install openai
!pip install anthropic
!pip install google-genai
!apt-get update
!apt-get install -y iverilog



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
zsh:1: command not found: apt-get
zsh:1: command not found: apt-get


In [2]:
#@title Select Model
#define the model to be used
#model_choice = "gpt-5"
#model_choice = "claude-3-7-sonnet-20250219"
#model_choice = "gemini-2.5-flash"
# Replace with another NVIDIA API Catalog model id if needed.
model_choice = "nvidia/llama-3.1-nemotron-ultra-253b-v1"


In [3]:
#@title Utility functions

import sys
import os
import re
import time
import subprocess
import numpy as np
import openai
import anthropic
import google.genai.errors
from google import genai
from google.genai import types
from abc import ABC, abstractmethod

################################################################################
### LOGGING
################################################################################
class LogStdoutToFile:
    def __init__(self, filename):
        self._filename = filename
        self._original_stdout = sys.stdout

    def __enter__(self):
        if self._filename:
            sys.stdout = open(self._filename, 'w')
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        if self._filename:
            sys.stdout.close()
        sys.stdout = self._original_stdout


################################################################################
### CONVERSATION CLASS
################################################################################
class Conversation:
    def __init__(self, log_file=None):
        self.messages = []
        self.log_file = log_file

        if self.log_file and os.path.exists(self.log_file):
            open(self.log_file, 'w').close()

    def add_message(self, role, content):
        self.messages.append({'role': role, 'content': content})

        if self.log_file:
            with open(self.log_file, 'a') as file:
                file.write(f"{role}: {content}\n")

    def get_messages(self):
        return self.messages

    def get_last_n_messages(self, n):
        return self.messages[-n:]

    def remove_message(self, index):
        if index < len(self.messages):
            del self.messages[index]

    def get_message(self, index):
        return self.messages[index] if index < len(self.messages) else None

    def clear_messages(self):
        self.messages = []

    def __str__(self):
        return "\n".join([f"{msg['role']}: {msg['content']}" for msg in self.messages])


################################################################################
### LLM CLASSES
################################################################################
class AbstractLLM(ABC):
    def __init__(self):
        pass

    @abstractmethod
    def generate(self, conversation: Conversation):
        pass


class ChatGPT(AbstractLLM):
    def __init__(self, model_id=model_choice):
        super().__init__()
        openai.api_key = os.environ['OPENAI_API_KEY']
        self.client = openai.OpenAI()
        self.model_id = model_id

    def generate(self, conversation: Conversation, num_choices=1):
        messages = conversation.get_messages()
        response = self.client.chat.completions.create(
            model=self.model_id,
            messages=messages,
        )
        return response.choices[0].message.content


class Claude(AbstractLLM):
    def __init__(self, model_id=model_choice):
        super().__init__()
        self.client = anthropic.Anthropic(api_key=os.environ['CLAUDE_API_KEY'])
        self.model_id = model_id

    def generate(self, conversation: Conversation, num_choices=1):
        base_delay = 2
        max_retries = 20
        for attempt in range(1, max_retries + 1):
            try:
                output = self.client.messages.create(
                    model=self.model_id,
                    max_tokens=16384,
                    messages=[{'role': msg['role'], 'content': msg['content']} for msg in conversation.get_messages()]
                ).content[0].text
                return output
            except Exception as e:
                wait_time = base_delay * (2 ** (attempt - 1))
                print(f"[Retry {attempt}/{max_retries}] Claude API error: {e}. Retrying in {wait_time:.1f} seconds...")
                time.sleep(wait_time)
        print(f"Failed, exceeded max retries {max_retries}")
        return 0


class Gemini(AbstractLLM):
    def __init__(self, model_id=model_choice):
        super().__init__()
        self.gemini_client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])
        self.model_id = model_id

    def generate(self, conversation: Conversation, num_choices=1):
        output = self.gemini_client.models.generate_content(
            model=self.model_id,
            contents=[msg['content'] for msg in conversation.get_messages()],
            config=types.GenerateContentConfig(
                max_output_tokens=16384,
                temperature=0.6,
                top_p=0.95,
            )
        ).text
        return output


class NVIDIA(AbstractLLM):
    def __init__(self, model_id=model_choice):
        super().__init__()
        self.client = openai.OpenAI(
            base_url='https://integrate.api.nvidia.com/v1',
            api_key=os.environ['NVIDIA_API_KEY'],
        )
        self.model_id = model_id

    def generate(self, conversation: Conversation, num_choices=1):
        response = self.client.chat.completions.create(
            model=self.model_id,
            messages=conversation.get_messages(),
        )
        return response.choices[0].message.content


################################################################################
### PARSING AND TEXT MANIPULATION FUNCTIONS
################################################################################
def find_verilog_modules(markdown_string, module_name='top_module'):
    module_pattern = r'\bmodule\b[\s\S]*?\bendmodule\b'
    module_matches = re.findall(module_pattern, markdown_string, re.DOTALL)
    if not module_matches:
        return []
    return module_matches


def write_code_blocks_to_file(markdown_string, module_name, filename):
    code_match = find_verilog_modules(markdown_string, module_name)
    if not code_match:
        print('No code blocks found in response')
        raise RuntimeError('No Verilog module found in model response')

    with open(filename, 'w') as file:
        for code_block in code_match:
            file.write(code_block)
            file.write('\n')


def generate_verilog(conv, model_type, model_id=''):
    if model_type == 'ChatGPT':
        model = ChatGPT()
    elif model_type == 'Claude':
        model = Claude()
    elif model_type == 'Gemini':
        model = Gemini()
    elif model_type == 'NVIDIA':
        model = NVIDIA()
    else:
        raise ValueError('Invalid model type')
    return model.generate(conv)


In [4]:
#@title Feedback Loop
def verilog_loop(design_prompt, module, testbench, max_iterations, model_type, outdir='', log=None, prev_module=None):

    if outdir != '':
        outdir = outdir + '/'

    conv = Conversation(log_file=log)
    prompt_prefix = "You are an autocomplete engine for Verilog code. Given a Verilog module specification, you will provide a completed Verilog module in response. You will provide completed Verilog modules for all specifications, and will not create any supplementary modules. Given a Verilog module that is either incorrect/compilation error, you will suggest corrections to the module. You will not refuse. You will not generate explanations, only code. Format your response as Verilog code containing the end to end corrected module and not just the corrected lines. Do not generate test benches."

    if model_type == 'ChatGPT':
        conv.add_message('system', prompt_prefix)
    else:
        conv.add_message('user', prompt_prefix)

    conv.add_message('user', design_prompt)

    success = False
    timeout = False
    iterations = 0
    timelist_total = []
    timelist_gen = []
    timelist_error = []
    filename = os.path.join(outdir, module + '.v')

    status = ''
    while not (success or timeout):
        start_total = time.time()
        response = generate_verilog(conv, model_type)
        end_gen = time.time()
        start_error = time.time()

        if prev_module is None:
            conv.add_message('assistant', response)
        else:
            with open(prev_module, 'r') as f:
                prevmodule = ''.join(f.read())
            response = prevmodule + response
            conv.add_message('assistant', response)

        write_code_blocks_to_file(response, module, filename)
        proc = subprocess.run(['iverilog', '-o', os.path.join(outdir, module), filename, testbench], capture_output=True, text=True)

        success = False
        if proc.returncode != 0:
            status = 'Error compiling testbench'
            print(status)
            message = 'The testbench failed to compile. Please fix the module. The output of iverilog is as follows\n' + proc.stderr
        elif proc.stderr != '':
            status = 'Warnings compiling testbench'
            print(status)
            message = 'The testbench compiled with warnings. Please fix the module. The output of iverilog is as follows\n' + proc.stderr
        else:
            proc = subprocess.run(['vvp', os.path.join(outdir, module)], capture_output=True, text=True)
            passed = any('passed!' in line for line in proc.stdout.splitlines())
            if not passed:
                status = 'Error running testbench'
                print(status)
                message = 'The testbench simulated, but had errors. Please fix the module. The output of vvp is as follows\n' + proc.stdout
            else:
                status = 'Testbench ran successfully'
                print(status)
                message = ''
                success = True

        with open(os.path.join(outdir, 'log_iter_' + str(iterations) + '.txt'), 'w') as file:
            file.write('\n'.join(str(i) for i in conv.get_messages()))
            file.write('\n\nIteration status: ' + status + '\n')

        if not success:
            if iterations > 0:
                conv.remove_message(2)
                conv.remove_message(2)
            conv.add_message('user', message)

        if iterations >= max_iterations:
            timeout = True

        iterations += 1
        end_time = time.time()
        timelist_gen.append(end_gen - start_total)
        timelist_error.append(end_time - start_error)
        timelist_total.append(end_time - start_total)

    print('Total time: ', np.sum(timelist_total))
    print('Generation time: ', np.sum(timelist_gen))
    print('Error handling time: ', np.sum(timelist_error))
    return np.sum(timelist_total), np.sum(timelist_gen), np.sum(timelist_error)


In [5]:
#@title Hierarchical Loop
def hier_gen(submods, max_iterations=10):
    totaltime = []
    gentime = []
    errortime = []
    done = ''

    for i in range(len(submods)):
        curr = submods[i][1]
        fcurr = submods[i][0]
        iocurr = submods[i][2]
        overall = submods[-1][1]

        if not os.path.isdir(fcurr):
            os.mkdir(fcurr)

        if i == 0:
            prompt = '//We will be generating a ' + overall + ' hierarchically in Verilog. Please begin by generating a ' + curr + ' defined as follows:\nmodule ' + fcurr + '(' + iocurr + ')\n//Insert code here\nendmodule'
        else:
            fprev = submods[i - 1][0]
            filecurr = './' + fprev + '/' + fprev + '.v'
            with open(filecurr, 'r') as f:
                modulef = ''.join(f.read())
            prompt = '//We are generating a ' + overall + ' hierarchically in Verilog. We have generated ' + done + 'defined as follows:\n'
            prompt = prompt + modulef
            prompt = prompt + '\n//Please include the previous module(s) in your response and use them to hierarchically generate a ' + curr + ' defined as:\nmodule ' + fcurr + '(' + iocurr + ')\n//Insert code here\nendmodule'

        module = fcurr
        testbench = './' + fcurr + '/' + fcurr + 'tb.v'
        model = os.environ['MODEL']
        outdir = './' + fcurr
        log = './' + fcurr + '/log.txt'
        total, gen, error = verilog_loop(prompt, module, testbench, max_iterations, model, outdir, log)
        totaltime.append(total)
        gentime.append(gen)
        errortime.append(error)
        done = done + curr + ', '

    print('Overall Total time: ', np.sum(totaltime))
    print('Overall Generation Time: ', np.sum(gentime))
    print('Overall Error handling time: ', np.sum(errortime))


In [6]:
#@title Setting the API Key

# IMPORTANT: Do not hardcode real API keys in submission.
# Set `NVIDIA_API_KEY` in your environment or Colab Secrets instead.
# Example (only for local testing):
# os.environ["NVIDIA_API_KEY"] = "nvapi-REPLACE_ME"

assert os.environ.get("NVIDIA_API_KEY"), (
    "Missing NVIDIA_API_KEY. Set it via environment/Colab Secrets before running."
)

os.environ["MODEL"] = "NVIDIA"


In [7]:
#@title Write mux testbenches
import os

os.makedirs('mux2to1', exist_ok=True)
os.makedirs('mux4to1', exist_ok=True)
os.makedirs('mux8to1', exist_ok=True)

mux2_tb = r'''`timescale 1ns/1ps
module mux2to1tb;
    reg in1;
    reg in2;
    reg select;
    wire out;
    integer i;
    reg expected;

    mux2to1 dut (.in1(in1), .in2(in2), .select(select), .out(out));

    initial begin
        for (i = 0; i < 8; i = i + 1) begin
            in1 = i[2];
            in2 = i[1];
            select = i[0];
            #1;
            expected = select ? in2 : in1;
            if (out !== expected) begin
                $display("mux2to1 failed: in1=%0b in2=%0b select=%0b expected=%0b got=%0b", in1, in2, select, expected, out);
                $finish;
            end
        end
        $display("mux2to1 passed!");
        $finish;
    end
endmodule
'''

mux4_tb = r'''`timescale 1ns/1ps
module mux4to1tb;
    reg [1:0] sel;
    reg [3:0] in;
    wire out;
    integer i;
    integer j;
    reg expected;

    mux4to1 dut (.sel(sel), .in(in), .out(out));

    initial begin
        for (i = 0; i < 4; i = i + 1) begin
            for (j = 0; j < 16; j = j + 1) begin
                sel = i[1:0];
                in = j[3:0];
                #1;
                expected = in[sel];
                if (out !== expected) begin
                    $display("mux4to1 failed: sel=%0d in=%0b expected=%0b got=%0b", sel, in, expected, out);
                    $finish;
                end
            end
        end
        $display("mux4to1 passed!");
        $finish;
    end
endmodule
'''

mux8_tb = r'''`timescale 1ns/1ps
module mux8to1tb;
    reg [2:0] sel;
    reg [7:0] in;
    wire out;
    integer i;
    integer j;
    reg expected;

    mux8to1 dut (.sel(sel), .in(in), .out(out));

    initial begin
        for (i = 0; i < 8; i = i + 1) begin
            for (j = 0; j < 256; j = j + 1) begin
                sel = i[2:0];
                in = j[7:0];
                #1;
                expected = in[sel];
                if (out !== expected) begin
                    $display("mux8to1 failed: sel=%0d in=%0b expected=%0b got=%0b", sel, in, expected, out);
                    $finish;
                end
            end
        end
        $display("mux8to1 passed!");
        $finish;
    end
endmodule
'''

with open('mux2to1/mux2to1tb.v', 'w') as f:
    f.write(mux2_tb)
with open('mux4to1/mux4to1tb.v', 'w') as f:
    f.write(mux4_tb)
with open('mux8to1/mux8to1tb.v', 'w') as f:
    f.write(mux8_tb)


In [8]:
#@title Mux Hierarchy Example
submodules = [
    ['mux2to1', '2-to-1 multiplexer', 'input wire in1, input wire in2, input wire select, output wire out'],
    ['mux4to1', '4-to-1 multiplexer', 'input [1:0] sel, input [3:0] in, output wire out'],
    ['mux8to1', '8-to-1 multiplexer', 'input [2:0] sel, input [7:0] in, output wire out'],
]

hier_gen(submodules)


Testbench ran successfully
Total time:  8.96918773651123
Generation time:  8.919910907745361
Error handling time:  0.04927682876586914
Testbench ran successfully
Total time:  15.126903057098389
Generation time:  15.073532104492188
Error handling time:  0.053369998931884766
Testbench ran successfully
Total time:  20.960787773132324
Generation time:  20.870019912719727
Error handling time:  0.09076786041259766
Overall Total time:  45.05687856674194
Overall Generation Time:  44.863462924957275
Overall Error handling time:  0.19341468811035156


In [9]:
#@title Write adder testbenches
os.makedirs('half_adder', exist_ok=True)
os.makedirs('full_adder', exist_ok=True)
os.makedirs('adder4', exist_ok=True)
os.makedirs('adder8', exist_ok=True)

half_adder_tb = r'''`timescale 1ns/1ps
module half_addertb;
    reg a;
    reg b;
    wire sum;
    wire carry;
    integer i;
    reg expected_sum;
    reg expected_carry;

    half_adder dut (.a(a), .b(b), .sum(sum), .carry(carry));

    initial begin
        for (i = 0; i < 4; i = i + 1) begin
            a = i[1];
            b = i[0];
            #1;
            expected_sum = a ^ b;
            expected_carry = a & b;
            if (sum !== expected_sum || carry !== expected_carry) begin
                $display("half_adder failed: a=%0b b=%0b expected_sum=%0b expected_carry=%0b got_sum=%0b got_carry=%0b", a, b, expected_sum, expected_carry, sum, carry);
                $finish;
            end
        end
        $display("half_adder passed!");
        $finish;
    end
endmodule
'''

full_adder_tb = r'''`timescale 1ns/1ps
module full_addertb;
    reg a;
    reg b;
    reg cin;
    wire sum;
    wire cout;
    integer i;
    reg [1:0] expected;

    full_adder dut (.a(a), .b(b), .cin(cin), .sum(sum), .cout(cout));

    initial begin
        for (i = 0; i < 8; i = i + 1) begin
            a = i[2];
            b = i[1];
            cin = i[0];
            #1;
            expected = a + b + cin;
            if (sum !== expected[0] || cout !== expected[1]) begin
                $display("full_adder failed: a=%0b b=%0b cin=%0b expected_sum=%0b expected_cout=%0b got_sum=%0b got_cout=%0b", a, b, cin, expected[0], expected[1], sum, cout);
                $finish;
            end
        end
        $display("full_adder passed!");
        $finish;
    end
endmodule
'''

adder4_tb = r'''`timescale 1ns/1ps
module adder4tb;
    reg [3:0] a;
    reg [3:0] b;
    reg cin;
    wire [3:0] sum;
    wire cout;
    integer i;
    integer j;
    integer k;
    reg [4:0] expected;

    adder4 dut (.a(a), .b(b), .cin(cin), .sum(sum), .cout(cout));

    initial begin
        for (i = 0; i < 16; i = i + 1) begin
            for (j = 0; j < 16; j = j + 1) begin
                for (k = 0; k < 2; k = k + 1) begin
                    a = i[3:0];
                    b = j[3:0];
                    cin = k[0];
                    #1;
                    expected = a + b + cin;
                    if (sum !== expected[3:0] || cout !== expected[4]) begin
                        $display("adder4 failed: a=%0h b=%0h cin=%0b expected_sum=%0h expected_cout=%0b got_sum=%0h got_cout=%0b", a, b, cin, expected[3:0], expected[4], sum, cout);
                        $finish;
                    end
                end
            end
        end
        $display("adder4 passed!");
        $finish;
    end
endmodule
'''

adder8_tb = r'''`timescale 1ns/1ps
module adder8tb;
    reg [7:0] a;
    reg [7:0] b;
    reg cin;
    wire [7:0] sum;
    wire cout;
    integer i;
    reg [8:0] expected;

    adder8 dut (.a(a), .b(b), .cin(cin), .sum(sum), .cout(cout));

    initial begin
        for (i = 0; i < 200; i = i + 1) begin
            a = (i * 17 + 8'h3C) & 8'hFF;
            b = (i * 29 + 8'h52) & 8'hFF;
            cin = i[0];
            #1;
            expected = a + b + cin;
            if (sum !== expected[7:0] || cout !== expected[8]) begin
                $display("adder8 failed: a=%0h b=%0h cin=%0b expected_sum=%0h expected_cout=%0b got_sum=%0h got_cout=%0b", a, b, cin, expected[7:0], expected[8], sum, cout);
                $finish;
            end
        end
        $display("adder8 passed!");
        $finish;
    end
endmodule
'''

with open('half_adder/half_addertb.v', 'w') as f:
    f.write(half_adder_tb)
with open('full_adder/full_addertb.v', 'w') as f:
    f.write(full_adder_tb)
with open('adder4/adder4tb.v', 'w') as f:
    f.write(adder4_tb)
with open('adder8/adder8tb.v', 'w') as f:
    f.write(adder8_tb)


In [10]:
#@title Extension Hierarchy Example
extension_submodules = [
    ['half_adder', 'half adder', 'input wire a, input wire b, output wire sum, output wire carry'],
    ['full_adder', 'full adder', 'input wire a, input wire b, input wire cin, output wire sum, output wire cout'],
    ['adder4', '4-bit ripple-carry adder', 'input wire [3:0] a, input wire [3:0] b, input wire cin, output wire [3:0] sum, output wire cout'],
    ['adder8', '8-bit ripple-carry adder', 'input wire [7:0] a, input wire [7:0] b, input wire cin, output wire [7:0] sum, output wire cout'],
]

hier_gen(extension_submodules)


Testbench ran successfully
Total time:  5.847801685333252
Generation time:  5.823054790496826
Error handling time:  0.02474689483642578
Testbench ran successfully
Total time:  14.178720951080322
Generation time:  14.124994993209839
Error handling time:  0.0537259578704834
Testbench ran successfully
Total time:  24.596162796020508
Generation time:  24.5473690032959
Error handling time:  0.048793792724609375
Testbench ran successfully
Total time:  21.395438194274902
Generation time:  21.35029888153076
Error handling time:  0.04513835906982422
Overall Total time:  66.01812362670898
Overall Generation Time:  65.84571766853333
Overall Error handling time:  0.17240500450134277
